# Modeling — **SVM + TF-IDF (sklearn)** (Colab) — *driver*

Pendamping notebook IndoBERT. Driver tipis: clone repo → jalankan
`src.modeling.train_svm_full14k` (fitur IDENTIK dgn .py: GridSearch 24 kombinasi
ngram×min_df×C, split kanonik 70/20/10 seed42, metrik + confusion matrix).
Pilih dataset di **bagian 2** via `TAG`/`SUBSET` (default **balanced3k**).

> CPU cukup (cepat). **SVM Spark MLlib** sengaja TIDAK di sini — itu jalur Big Data,
> dijalankan di cluster (lihat repo `src/spark/`), bukan single-node Colab.

## 1. Clone repo (privat) + dependensi

In [ ]:
import os
from getpass import getpass
GH_TOKEN = os.environ.get('GH_TOKEN') or getpass('GitHub PAT (classic, scope repo): ')
![ -d jokowi_sentiment_project ] || git clone https://{GH_TOKEN}@github.com/Arachnoida/jokowi_sentiment_project.git
%cd jokowi_sentiment_project
%pip install -q 'pymongo>=4' dnspython certifi scikit-learn matplotlib pandas numpy python-dotenv

## 2. Set MONGO_URI + jalankan training
`raw_comments` (label) + `processed_svm` (teks `svm` stemmed) dibaca dari Mongo Atlas
(butuh IP allowlist `0.0.0.0/0`). Subset di-filter ke 3000 baris → split 2100/600/300.

In [ ]:
import os
from getpass import getpass
os.environ['MONGO_URI'] = os.environ.get('MONGO_URI') or getpass('MONGO_URI: ')

# === Pilih dataset ===
#   balanced 3k : TAG, SUBSET = 'balanced3k', 'outputs/labeling/balanced_3000.csv'
#   full 14k    : TAG, SUBSET = 'full14k', ''
TAG    = 'balanced3k'
SUBSET = 'outputs/labeling/balanced_3000.csv'

flags = f'--tag {TAG}' + (f' --subset {SUBSET}' if SUBSET else '')
!python -m src.modeling.train_svm_full14k $flags

## 3. Lihat hasil + confusion matrix

In [ ]:
import json
from IPython.display import Image, display
m = json.load(open(f'outputs/reports/svm_{TAG}_metrics.json'))['test']
print('Akurasi :', m['accuracy'])
print('macro-F1:', m['macro_f1'])
for l in ['Negatif','Netral','Positif']:
    print(f"  {l:<8} F1={m['per_class'][l]['f1']:.3f} (n={m['per_class'][l]['support']})")
print('best params:', m.get('best_params'))
display(Image(f'outputs/reports/svm_{TAG}_confusion.png'))

## 4. (Opsional) Bandingkan 3 model
Jika `indobert_<TAG>_metrics.json` & `svm_spark_<TAG>_metrics.json` sudah ada di
`outputs/reports/`, hasilkan tabel + chart akurasi 3 model.

In [ ]:
suffix = '' if TAG == 'full14k' else f'_{TAG}'
import os
need = [f'outputs/reports/svm_{TAG}_metrics.json',
        f'outputs/reports/svm_spark{suffix}_metrics.json',
        f'outputs/reports/indobert{suffix}_metrics.json']
if all(os.path.exists(p) for p in need):
    !python -m src.modeling.compare_models --tag $TAG
else:
    print('Lewati: butuh ketiga JSON. Tersedia:', [p for p in need if os.path.exists(p)])

In [ ]:
# (Opsional) unduh artefak SVM untuk di-commit ke outputs/reports/
from google.colab import files
files.download(f'outputs/reports/svm_{TAG}_metrics.json')
files.download(f'outputs/reports/svm_{TAG}_confusion.png')